In [ ]:
# Install the core LangChain and LangGraph ecosystem
!pip install -qU langchain langgraph langchain-core langchain-community

# Install open-source vector DB and local embeddings (Required by technical constraints)
!pip install -qU faiss-cpu sentence-transformers langchain-huggingface

# Install data ingestion tools
!pip install -qU yfinance feedparser pypdf

# Install LLM backbone (Using Groq for fast, free access to Llama 3 - Open Source compliance)
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 

In [17]:
!pip install -qU streamlit yfinance langchain-community langchain-text-splitters langchain-huggingface pypdf faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 50.0 MB/s eta 0:00:00


In [28]:
import os
from google.colab import userdata

# Load API Key
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("API Key loaded successfully!")
except Exception as e:
    print("Please set your GROQ_API_KEY in the Colab Secrets tab.")

# --- LLM Initialization ---
from langchain_groq import ChatGroq

# We declare our LLM backbone here.
# Upgraded to Llama 3.3 70B, which is the current active flagship on Groq
# --- INITIALIZE LLM ---
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1)

API Key loaded successfully!


In [ ]:
from typing import TypedDict, List, Annotated, Dict, Any
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

# 1. Define the final output artifact structure using Pydantic
class DecisionArtifact(BaseModel):
    asset: str = Field(description="The ticker symbol or market sector analyzed")
    action: str = Field(description="Action to take: BUY, SELL, HOLD, or WATCH")
    confidence_level: str = Field(description="High, Medium, or Low")
    evidence: List[str] = Field(description="2-3 bullet points of supporting evidence")
    risk_flags: List[str] = Field(description="Contradictory signals or data gaps")

# 2. Define the unified State that all agents will share
class AgentState(TypedDict):
    # --- Ingestion Data ---
    raw_input: str
    input_type: str                  # "ticker", "news", or "document"
    normalized_signal: Dict[str, Any] # Cleaned data with timestamp and core metrics

    # --- RAG Context ---
    retrieved_contexts: List[Dict[str, str]] # List of dicts containing 'text' and 'source'

    # --- Reasoning Data ---
    classification: str              # "bullish", "bearish", "neutral"
    hypothesis: str                  # The generated reasoning
    confidence_score: int            # 0 to 100

    # --- Final Output ---
    decision_artifact: Dict[str, Any]

    # --- Observability ---
    # add_messages ensures logs append rather than overwrite
    agent_logs: Annotated[List[str], add_messages]
    errors: List[str]

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

class RAGRetrievalAgent:
    def __init__(self):
        print("Initializing local open-source embedding model...")
        # Local, open-source embedding model as strictly mandated by constraints
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )
        self.vector_store = None

    def ingest_documents(self, texts: list[str], metadatas: list[dict] = None):
        """Chunks, embeds, and loads text data into the FAISS vector database."""
        docs = []
        for i, text in enumerate(texts):
            meta = metadatas[i] if metadatas else {"source": "unknown_document"}
            chunks = self.text_splitter.split_text(text)
            for chunk in chunks:
                docs.append(Document(page_content=chunk, metadata=meta))

        if self.vector_store is None:
            self.vector_store = FAISS.from_documents(docs, self.embeddings)
        else:
            self.vector_store.add_documents(docs)

        return len(docs) # Returns total chunk count for UI observability

    def retrieve_context(self, query: str, k: int = 3) -> list[dict]:
        """Queries the vector database and returns top-k hits with source tracking."""
        if self.vector_store is None:
            return [{"text": "Knowledge base empty.", "source": "system"}]

        results = self.vector_store.similarity_search(query, k=k)

        formatted_results = []
        for doc in results:
            formatted_results.append({
                "text": doc.page_content,
                "source": doc.metadata.get("source", "unknown")
            })
        return formatted_results

# --- Instantiate the RAG Agent Global Instance ---
rag_agent = RAGRetrievalAgent()

# Let's seed it with sample financial data for testing
sample_reports = [
    "Apple Inc. (AAPL) Q3 2026 Earnings: Record service revenue growth at 14% year-over-year. iPhone sales stable, but hardware margins compressed slightly due to supply chain input costs in Asia.",
    "NVIDIA (NVDA) Technology Update 2026: Demand for next-generation B200 and Ultra chips continues to outpace production supply. High enterprise spending confirmed across major cloud providers."
]
sample_meta = [{"source": "AAPL_Q3_2026.txt"}, {"source": "NVDA_Update_2026.txt"}]

chunks_created = rag_agent.ingest_documents(sample_reports, sample_meta)
print(f"RAG Engine active. Successfully ingested sample documents into {chunks_created} vector chunks.")


/tmp/ipykernel_939/3195510134.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Initializing local open-source embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

RAG Engine active. Successfully ingested sample documents into 2 vector chunks.


In [ ]:
import datetime

class SignalIngestionAgent:
    @staticmethod
    def process(raw_input: str, input_type: str) -> dict:
        """Parses multi-modal data streams and transforms them into a clean internal format."""
        timestamp = datetime.datetime.utcnow().isoformat() + "Z"

        normalized = {
            "timestamp": timestamp,
            "asset": "UNKNOWN",
            "metrics": {},
            "raw_text_summary": ""
        }

        try:
            if input_type == "ticker":
                # Expected format format: "TICKER,PRICE,VOLUME,RSI" (e.g. "AAPL,185.50,1200000,62")
                parts = [p.strip() for p in raw_input.split(",")]
                normalized["asset"] = parts[0]
                normalized["metrics"] = {
                    "price": float(parts[1]),
                    "volume": int(parts[2]),
                    "rsi": float(parts[3]) if len(parts) > 3 else 50.0
                }
                normalized["raw_text_summary"] = f"Price tick update for {parts[0]} at ${parts[1]}."

            elif input_type == "news":
                # Expected format: "ASSET | Headline text summary"
                if "|" in raw_input:
                    asset_part, headline = raw_input.split("|", 1)
                    normalized["asset"] = asset_part.strip()
                    normalized["raw_text_summary"] = headline.strip()
                else:
                    normalized["raw_text_summary"] = raw_input

            elif input_type == "document":
                # General text reference context update
                normalized["raw_text_summary"] = raw_input[:200] + "..." # Truncate for display logging

            else:
                raise ValueError(f"Unsupported signal type: {input_type}")

            return normalized

        except Exception as e:
            # Drop clean exception context to bubble up to LangGraph's error state array
            raise ValueError(f"Signal normalization engine failed: {str(e)}")

In [ ]:
# Simulated incoming data packet
raw_tick = "NVDA, 132.40, 4500000, 72.5"
normalized_tick = SignalIngestionAgent.process(raw_tick, "ticker")

print("--- Normalized Output ---")
print(normalized_tick)

# Try pulling context from our vector store based on the normalized output asset target
print("\n--- RAG Fetch Test ---")
context_found = rag_agent.retrieve_context(query=f"Supply chain update for {normalized_tick['asset']}")
print(context_found)

--- Normalized Output ---
{'timestamp': '2026-07-19T10:00:39.911486Z', 'asset': 'NVDA', 'metrics': {'price': 132.4, 'volume': 4500000, 'rsi': 72.5}, 'raw_text_summary': 'Price tick update for NVDA at $132.40.'}

--- RAG Fetch Test ---
[{'text': 'NVIDIA (NVDA) Technology Update 2026: Demand for next-generation B200 and Ultra chips continues to outpace production supply. High enterprise spending confirmed across major cloud providers.', 'source': 'NVDA_Update_2026.txt'}, {'text': 'Apple Inc. (AAPL) Q3 2026 Earnings: Record service revenue growth at 14% year-over-year. iPhone sales stable, but hardware margins compressed slightly due to supply chain input costs in Asia.', 'source': 'AAPL_Q3_2026.txt'}]


/tmp/ipykernel_939/2491535532.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().isoformat() + "Z"


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
import json

# ==========================================
# 1. DEFINE AGENT NODES (DEFENSIVE VERSION)
# ==========================================

class AnalysisOutput(BaseModel):
    classification: str = Field(description="Must be exactly one of: bullish, bearish, or neutral")
    hypothesis: str = Field(description="A concise statement outlining the core opportunity or risk, supporting signals, and expected time horizon.")
    confidence_score: int = Field(description="An integer from 0 to 100 assessing the structural strength of the evidence.")

def analysis_node(state: AgentState) -> dict:
    print("\n[Agent] Signal Analysis & Hypothesis Agent activated...")
    signal = state.get("normalized_signal", {})
    contexts = state.get("retrieved_contexts", [])

    context_str = "\n".join([
        f"- Reference Material from [{c['source']}]: {c['text']}"
        for c in contexts
    ])

    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are an elite hedge fund quantitative analysis agent. Your objective is to process incoming real-time "
            "market signals against historical background reference documents.\n\n"
            "Analyze the inputs critically, look for contradictions, verify structural signals, and populate the required output schema."
        )),
        ("user", (
            "INCOMING NORMALIZED SIGNAL:\n{signal}\n\n"
            "RETRIEVED KNOWLEDGE BASE CONTEXT:\n{context_str}\n\n"
            "Execute analysis and deliver structured output response matching the target schema specifications."
        ))
    ])

    structured_analyst = llm.with_structured_output(AnalysisOutput)
    analysis_chain = analysis_prompt | structured_analyst

    try:
        analysis_result = analysis_chain.invoke({
            "signal": str(signal),
            "context_str": context_str
        })
        log_entry = f"Analysis Agent completed. Action: {analysis_result.classification.upper()} (Confidence: {analysis_result.confidence_score})"
        return {
            "classification": analysis_result.classification,
            "hypothesis": analysis_result.hypothesis,
            "confidence_score": analysis_result.confidence_score,
            "agent_logs": [log_entry]
        }
    except Exception as e:
        error_msg = f"LLM / Analysis Node failed: {str(e)}"
        return {"errors": [error_msg], "agent_logs": [f"ERROR: {error_msg}"]}


def synthesis_node(state: AgentState) -> dict:
    print("\n[Agent] Decision Synthesis & Alerting Agent activated...")

    # Use .get() with safe defaults to prevent KeyErrors if previous nodes failed
    signal = state.get("normalized_signal", {})
    classification = state.get("classification", "UNKNOWN")
    hypothesis = state.get("hypothesis", "No hypothesis available due to upstream error.")
    confidence = state.get("confidence_score", 0)

    synthesis_prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a Senior Investment Portfolio Committee Engine. Your task is to ingest internal research "
            "hypotheses and format them into clear, production-grade actionable decision artifacts.\n\n"
            "Ensure the recommended action is explicit (BUY, SELL, HOLD, or WATCH) and capture critical risk flags "
            "if there are underlying data gaps or conflicting signals."
        )),
        ("user", (
            "TARGET ASSET: {asset}\n"
            "ANALYST CLASSIFICATION: {classification}\n"
            "PROPOSED HYPOTHESIS: {hypothesis}\n"
            "ANALYST CONFIDENCE SCORE: {confidence}\n\n"
            "Synthesize the final technical decision artifact ticket."
        ))
    ])

    structured_synthesizer = llm.with_structured_output(DecisionArtifact)
    synthesis_chain = synthesis_prompt | structured_synthesizer

    try:
        artifact = synthesis_chain.invoke({
            "asset": signal.get("asset", "UNKNOWN"),
            "classification": classification,
            "hypothesis": hypothesis,
            "confidence": confidence
        })

        log_entry = f"Synthesis Agent finalized ticket for {artifact.asset}. Action Recommended: {artifact.action}."

        if confidence >= 70:
            print(f"\n🚨 [CRITICAL HIGH CONFIDENCE SYSTEM ALERT] 🚨\n"
                  f"Asset: {artifact.asset} -> RECOMMENDED ACTION: {artifact.action}\n"
                  f"Confidence: {confidence}%\n")

        return {
            "decision_artifact": artifact.model_dump(),
            "agent_logs": [log_entry]
        }
    except Exception as e:
        error_msg = f"LLM / Synthesis Node failed: {str(e)}"
        return {"errors": [error_msg], "agent_logs": [f"ERROR: {error_msg}"]}

# ==========================================
# 2. RUN DEFENSIVE VERIFICATION TEST
# ==========================================

mock_state = {
    "normalized_signal": normalized_tick,
    "retrieved_contexts": context_found,
    "agent_logs": [],
    "errors": []
}

# 1. Run Analysis
analysis_updates = analysis_node(mock_state)
mock_state.update(analysis_updates)

# 2. Check for errors BEFORE running Synthesis (This simulates LangGraph's Conditional Edges)
if mock_state.get("errors"):
    print(f"\n❌ PIPELINE HALTED: The Analysis Agent threw an error:\n>>> {mock_state['errors'][-1]}")
    print("\nTroubleshooting: If this is a Groq error, verify your GROQ_API_KEY is loaded correctly and you haven't hit the free-tier rate limit.")
else:
    # 3. Run Synthesis only if Analysis succeeded
    synthesis_updates = synthesis_node(mock_state)
    mock_state.update(synthesis_updates)

    if mock_state.get("errors"):
         print(f"\n❌ PIPELINE HALTED: The Synthesis Agent threw an error:\n>>> {mock_state['errors'][-1]}")
    else:
        print("\n--- Final Generated Investment Ticket Output ---")
        print(json.dumps(mock_state["decision_artifact"], indent=2))


[Agent] Signal Analysis & Hypothesis Agent activated...

[Agent] Decision Synthesis & Alerting Agent activated...

🚨 [CRITICAL HIGH CONFIDENCE SYSTEM ALERT] 🚨
Asset: NVDA -> RECOMMENDED ACTION: BUY
Confidence: 75%


--- Final Generated Investment Ticket Output ---
{
  "asset": "NVDA",
  "action": "BUY",
  "confidence_level": "Medium",
  "evidence": [
    "Strong fundamental demand for B200/Ultra chips is outpacing supply, supporting a sustained upside thesis.",
    "Projected price appreciation is expected over a 1-3 month horizon following a brief consolidation phase."
  ],
  "risk_flags": [
    "RSI at 72.5 indicates short-term overbought conditions, signaling potential for immediate price correction.",
    "Timing risk exists regarding the duration of the predicted consolidation period."
  ]
}


In [ ]:
from langgraph.graph import StateGraph, END

# ==========================================
# STEP 8: BUILD THE LANGGRAPH ORCHESTRATOR
# ==========================================

print("Assembling LangGraph Multi-Agent Orchestration Topology...")

# 1. Initialize the Graph using our unified state definition
workflow = StateGraph(AgentState)

# 2. Register Nodes (These are the functions we built in previous steps)
# We use lambda wrappers for the ingestion and RAG nodes to adapt them to the graph state signature
workflow.add_node("ingest_signal", lambda state: {
    "normalized_signal": SignalIngestionAgent.process(state["raw_input"], state["input_type"]),
    "agent_logs": ["Signal Ingestion completed successfully."]
})

workflow.add_node("retrieve_context", lambda state: {
    # Dynamically search the vector store based on the normalized asset target
    "retrieved_contexts": rag_agent.retrieve_context(f"General updates and risks for {state.get('normalized_signal', {}).get('asset', '')}"),
    "agent_logs": ["RAG Context Retrieval completed."]
})

workflow.add_node("analyze_hypothesis", analysis_node)
workflow.add_node("synthesize_decision", synthesis_node)

# We need an error handling node to catch any pipeline crashes gracefully
def error_handler(state: AgentState):
    print(f"\n[Error Node] Pipeline bypassed due to failure: {state.get('errors', ['Unknown Error'])[-1]}")
    return {"decision_artifact": {"asset": "ERROR", "action": "WATCH", "confidence_level": "Low", "evidence": [], "risk_flags": state.get('errors')}}

workflow.add_node("handle_error", error_handler)

# 3. Define the Flow Topology (Edges)
workflow.set_entry_point("ingest_signal")
workflow.add_edge("ingest_signal", "retrieve_context")
workflow.add_edge("retrieve_context", "analyze_hypothesis")

# 4. Conditional Edge Logic
# If the Analysis agent throws an error, route to the error handler instead of Synthesis
def route_after_analysis(state: AgentState) -> str:
    if state.get("errors"):
        return "handle_error"
    return "synthesize_decision"

workflow.add_conditional_edges(
    "analyze_hypothesis",
    route_after_analysis,
    {
        "synthesize_decision": "synthesize_decision",
        "handle_error": "handle_error"
    }
)

# Route everything to END when finished
workflow.add_edge("synthesize_decision", END)
workflow.add_edge("handle_error", END)

# 5. Compile the Engine
finagent_app = workflow.compile()
print("✅ LangGraph compiled successfully! Engine is ready for execution.")

Assembling LangGraph Multi-Agent Orchestration Topology...
✅ LangGraph compiled successfully! Engine is ready for execution.


In [ ]:
# Create a fresh starting state representing a real incoming market event
initial_event = {
    "raw_input": "AAPL | Supply chain constraints in Asia expected to impact Q4 margins.",
    "input_type": "news",
    "retrieved_contexts": [], # Graph will populate this
    "agent_logs": [],
    "errors": []
}

print("\n🚀 LAUNCHING FULL GRAPH EXECUTION...")

# .invoke() runs the graph synchronously (great for Colab testing)
final_state = finagent_app.invoke(initial_event)

print("\n--- Final Graph State Inspection ---")
print(f"Total Logs Captured: {len(final_state['agent_logs'])}")
print("\nGenerated Decision Artifact:")
import json
print(json.dumps(final_state["decision_artifact"], indent=2))


🚀 LAUNCHING FULL GRAPH EXECUTION...

[Agent] Signal Analysis & Hypothesis Agent activated...


/tmp/ipykernel_939/2491535532.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().isoformat() + "Z"



[Error Node] Pipeline bypassed due to failure: LLM / Analysis Node failed: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': ''}}

--- Final Graph State Inspection ---
Total Logs Captured: 3

Generated Decision Artifact:
{
  "asset": "ERROR",
  "action": "WATCH",
  "confidence_level": "Low",
  "evidence": [],
  "risk_flags": [
    "LLM / Analysis Node failed: Error code: 400 - {'error': {'message': \"Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.\", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': ''}}"
  ]
}


In [43]:
%%writefile app.py
import streamlit as st
import os
import datetime
import tempfile
import json
import random
import pandas as pd
import yfinance as yf
from typing import TypedDict, List, Annotated, Dict, Any
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# --- RAG IMPORTS ---
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- UI CONFIGURATION ---
st.set_page_config(page_title="FinAgent | Multi-Agent System", layout="wide")
st.title("📈 FinAgent: Real-Time Investment Orchestrator")

# --- SESSION STATE INITIALIZATION ---
if "history" not in st.session_state:
    st.session_state.history = []
if "vector_store" not in st.session_state:
    st.session_state.vector_store = None

# Cache the local embedding model
@st.cache_resource
def get_embeddings():
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embeddings = get_embeddings()

# --- SIDEBAR: SETTINGS & RAG UPLOADS ---
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input("Enter Groq API Key:", type="password")
    if api_key:
        os.environ["GROQ_API_KEY"] = api_key

    st.markdown("---")
    st.header("📂 1. Knowledge Base Management")
    uploaded_files = st.file_uploader("Upload Sector Reports (PDF/TXT)", accept_multiple_files=True)

    if st.button("🧠 Ingest Documents to FAISS"):
        if uploaded_files:
            with st.spinner("Chunking and Embedding documents..."):
                all_docs = []
                for file in uploaded_files:
                    with tempfile.NamedTemporaryFile(delete=False, suffix=f"_{file.name}") as tmp:
                        tmp.write(file.getvalue())
                        tmp_path = tmp.name

                    try:
                        if file.name.endswith(".pdf"):
                            loader = PyPDFLoader(tmp_path)
                        else:
                            loader = TextLoader(tmp_path)

                        docs = loader.load()
                        for d in docs:
                            d.metadata["source"] = file.name
                        all_docs.extend(docs)
                    finally:
                        os.unlink(tmp_path)

                if all_docs:
                    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                    splits = text_splitter.split_documents(all_docs)

                    if st.session_state.vector_store is None:
                        st.session_state.vector_store = FAISS.from_documents(splits, embeddings)
                    else:
                        st.session_state.vector_store.add_documents(splits)

                    st.success(f"✅ Embedded {len(splits)} chunks into Vector Store!")
        else:
            st.warning("Please upload a file first.")

    st.markdown("---")
    st.header("📡 2. Signal Feed Config")
    feed_source = st.selectbox("Market Data Source", ["API Ticker Symbol", "Simulated Stream", "File Path (CSV)"])
    alert_threshold = st.slider("High-Confidence Alert Threshold", 0, 100, 70)

if not api_key:
    st.warning("👈 Please enter your Groq API Key in the sidebar to start.")
    st.stop()

# --- INITIALIZE LLM ---
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1)

# --- LANGGRAPH STATE & SCHEMAS ---
class DecisionArtifact(BaseModel):
    asset: str = Field(description="The ticker symbol or market sector")
    action: str = Field(description="Exactly: BUY, SELL, HOLD, or WATCH")
    confidence_level: str = Field(description="High, Medium, or Low")
    confidence_score: int = Field(description="Score from 0 to 100")
    evidence: List[str] = Field(description="2-3 bullet points of evidence")
    risk_flags: List[str] = Field(description="Contradictory indicators or risks")

class AgentState(TypedDict):
    normalized_signal: Dict[str, Any]
    retrieved_contexts: List[Dict[str, str]]
    classification: str
    hypothesis: str
    confidence_score: int
    decision_artifact: Dict[str, Any]
    agent_logs: Annotated[List[str], add_messages]
    errors: List[str]

class AnalysisOutput(BaseModel):
    classification: str = Field(description="bullish, bearish, or neutral")
    hypothesis: str = Field(description="Concise statement of opportunity/risk")
    confidence_score: int = Field(description="Integer 0-100")

# --- AGENT NODES ---
def analysis_node(state: AgentState) -> dict:
    signal = state.get("normalized_signal", {})
    contexts = state.get("retrieved_contexts", [])

    if contexts:
        context_str = "\n".join([f"- [{c['source']}]: {c['text']}" for c in contexts])
    else:
        context_str = "No specific reference documents provided. Rely on intrinsic market knowledge."

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an elite quantitative analyst. Classify the market signal and provide a hypothesis based strictly on the current price and context."),
        ("user", "SIGNAL:\n{signal}\n\nCONTEXT:\n{context_str}")
    ])

    try:
        chain = prompt | llm.with_structured_output(AnalysisOutput)
        result = chain.invoke({"signal": str(signal), "context_str": context_str})
        return {
            "classification": result.classification,
            "hypothesis": result.hypothesis,
            "confidence_score": result.confidence_score,
            "agent_logs": [f"[AGENT: Analyst] Classification: {result.classification.upper()} | Confidence: {result.confidence_score} | Hypothesis formulated."]
        }
    except Exception as e:
        return {"errors": [f"[AGENT: Analyst] Error encountered: {str(e)}"]}

def synthesis_node(state: AgentState) -> dict:
    signal = state.get("normalized_signal", {})

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Portfolio Committee Engine. Synthesize the analyst's hypothesis into a final decision artifact."),
        ("user", "ASSET: {asset}\nCLASSIFICATION: {classification}\nHYPOTHESIS: {hypothesis}\nCONFIDENCE: {confidence}")
    ])

    try:
        chain = prompt | llm.with_structured_output(DecisionArtifact)
        artifact = chain.invoke({
            "asset": signal.get("asset", "UNKNOWN"),
            "classification": state.get("classification", ""),
            "hypothesis": state.get("hypothesis", ""),
            "confidence": state.get("confidence_score", 0)
        })
        return {
            "decision_artifact": artifact.model_dump(),
            "agent_logs": [f"[AGENT: Portfolio Synthesizer] Artifact constructed. Final Action Tool Called: {artifact.action}"]
        }
    except Exception as e:
        return {"errors": [f"[AGENT: Portfolio Synthesizer] Error encountered: {str(e)}"]}

# --- BUILD GRAPH ---
graph_builder = StateGraph(AgentState)
graph_builder.add_node("analyze", analysis_node)
graph_builder.add_node("synthesize", synthesis_node)
graph_builder.set_entry_point("analyze")
graph_builder.add_edge("analyze", "synthesize")
graph_builder.add_edge("synthesize", END)
fin_agent = graph_builder.compile()

# --- MAIN DASHBOARD LAYOUT (WITH TABS) ---
# ADDED THE 3RD TAB HERE
tab_dash, tab_debug, tab_chat = st.tabs(["📈 Live Dashboard", "🔍 RAG Debug & Search", "💬 Chat with FinAgent"])

with tab_dash:
    col1, col2 = st.columns([1, 1])

    with col1:
        st.subheader("📡 Live Signal Input")

        tickers_to_process = []
        prices_to_process = {}

        if feed_source == "API Ticker Symbol":
            asset_input = st.text_input("API Ticker Symbol(s) - Comma separated:", value="RELIANCE.NS, ZOMATO.NS, NVDA")
            st.info("🟢 **Live Mode Active:** Fetching real-time market valuations.")
            if asset_input:
                tickers_to_process = [t.strip().upper() for t in asset_input.split(",") if t.strip()]

        elif feed_source == "Simulated Stream":
            asset_input = st.text_input("Simulated Ticker:", value="NVDA")
            price_input = st.number_input("Simulated Price (₹ or $)", value=132.40)
            if asset_input:
                tickers_to_process = [asset_input.strip().upper()]
                prices_to_process[tickers_to_process[0]] = price_input

        elif feed_source == "File Path (CSV)":
            st.info("📂 Upload a CSV with a 'Ticker' column (and optional 'Price' column).")
            uploaded_signal_file = st.file_uploader("Upload CSV File", type=["csv"])
            if uploaded_signal_file:
                try:
                    df = pd.read_csv(uploaded_signal_file)
                    if 'Ticker' in df.columns:
                        for _, row in df.iterrows():
                            tick = str(row['Ticker']).strip().upper()
                            tickers_to_process.append(tick)
                            if 'Price' in df.columns and pd.notna(row['Price']):
                                prices_to_process[tick] = float(row['Price'])
                        st.success(f"Loaded {len(tickers_to_process)} tickers from file.")
                    else:
                        st.error("CSV format invalid. Ensure there is a 'Ticker' header.")
                except Exception as e:
                    st.error(f"Error reading file: {e}")

        if st.button("🚀 Run FinAgent Pipeline", type="primary", use_container_width=True):
            if not tickers_to_process:
                st.error("Please provide at least one valid ticker symbol or upload a valid CSV.")
            else:
                for current_asset in tickers_to_process:
                    with st.spinner(f"Processing {current_asset}..."):

                        dynamic_run_logs = []

                        if current_asset in prices_to_process:
                            current_price = prices_to_process[current_asset]
                            currency_symbol = "₹" if current_asset.endswith(".NS") or current_asset.endswith(".BO") else "$"
                            dynamic_run_logs.append(f"[SYSTEM] Local/CSV price utilized: {currency_symbol}{current_price}")
                        else:
                            actual_price = None
                            ticker_data = yf.Ticker(current_asset)
                            try:
                                actual_price = ticker_data.fast_info['last_price']
                            except Exception:
                                pass

                            if actual_price is None:
                                try:
                                    hist = ticker_data.history(period="1mo")
                                    if not hist.empty:
                                        actual_price = hist['Close'].iloc[-1]
                                except Exception:
                                    pass

                            if actual_price is None or str(actual_price) == "nan":
                                st.toast(f"⚠️ Yahoo API blocked Colab IP. Using mock price for {current_asset}", icon="⚠️")
                                actual_price = random.uniform(100.50, 3000.00)
                                dynamic_run_logs.append(f"[SYSTEM FALLBACK EVENT] API Fetch Failed. Injected simulated proxy price: {actual_price:.2f}")
                            else:
                                dynamic_run_logs.append(f"[SYSTEM] Live Yahoo Finance API fetch successful.")

                            current_price = round(float(actual_price), 2)
                            currency_symbol = "₹" if current_asset.endswith(".NS") or current_asset.endswith(".BO") else "$"

                        retrieved_contexts = []
                        if st.session_state.vector_store is not None:
                            query = f"Market outlook, valuation analysis, and operational risks for {current_asset}"
                            docs = st.session_state.vector_store.similarity_search(query, k=2)
                            retrieved_contexts = [{"source": d.metadata.get("source", "Unknown"), "text": d.page_content} for d in docs]

                        if retrieved_contexts:
                            dynamic_run_logs.append(f"[TOOL: FAISS Vector DB] Retrieved {len(retrieved_contexts)} exact RAG passages:")
                            for idx, c in enumerate(retrieved_contexts):
                                snippet = c['text'][:120].replace('\n', ' ') + "..."
                                dynamic_run_logs.append(f"  -> Source {idx+1} Attribution: {c['source']} | Passage: {snippet}")
                        else:
                            dynamic_run_logs.append("[TOOL: FAISS Vector DB] 0 passages found. ZERO-SHOT FALLBACK: Relying entirely on pre-trained intrinsic data weights.")

                        initial_state = {
                            "normalized_signal": {"asset": current_asset, "price": f"{currency_symbol}{current_price}"},
                            "retrieved_contexts": retrieved_contexts,
                            "agent_logs": dynamic_run_logs,
                            "errors": []
                        }

                        final_state = fin_agent.invoke(initial_state)

                        if final_state.get("errors"):
                            st.error(f"Pipeline error on {current_asset}: {final_state['errors'][-1]}")
                        else:
                            artifact = final_state.get("decision_artifact", {})
                            artifact["timestamp"] = datetime.datetime.now().strftime("%H:%M:%S")
                            artifact["price_evaluated"] = f"{currency_symbol}{current_price}"

                            # WE NOW SAVE THE INTERNAL HYPOTHESIS SO THE CHATBOT CAN READ IT LATER
                            artifact["internal_hypothesis"] = final_state.get("hypothesis", "No hypothesis generated.")

                            raw_logs = final_state.get("agent_logs", [])
                            artifact["logs"] = [msg.content if hasattr(msg, 'content') else str(msg) for msg in raw_logs]

                            st.session_state.history.insert(0, artifact)

                            if artifact.get("confidence_score", 0) >= alert_threshold:
                                st.toast(f"🚨 HIGH CONFIDENCE ALERT: {artifact.get('action')} {current_asset}", icon="🚨")

                st.success("🎉 Processing Complete!")

    with col2:
        st.subheader("📋 Decision Artefacts Feed")
        if not st.session_state.history:
            st.info("No decisions generated yet. Run the pipeline!")

        for i, art in enumerate(st.session_state.history):
            action = art.get('action', 'WATCH')
            action_color = "🟢" if action == 'BUY' else "🔴" if action == 'SELL' else "🟠"
            is_high_confidence = art.get("confidence_score", 0) >= alert_threshold

            with st.container(border=True):
                if is_high_confidence:
                    if action == "BUY":
                        st.success(f"🚨 **HIGH CONFIDENCE DETECTED:** Strong Buy Signal for {art.get('asset')}")
                    elif action == "SELL":
                        st.error(f"🚨 **HIGH CONFIDENCE DETECTED:** Strong Sell Signal for {art.get('asset')}")
                    else:
                        st.warning(f"🚨 **HIGH CONFIDENCE DETECTED:** Watch/Hold Signal for {art.get('asset')}")

                st.markdown(f"### {art.get('asset')} {action_color} {action}")
                st.caption(f"Time: {art.get('timestamp')} | **Evaluated Value:** {art.get('price_evaluated')} | **Confidence:** {art.get('confidence_level')} ({art.get('confidence_score')}/100)")

                st.markdown("**Key Evidence:**")
                for bullet in art.get('evidence', []):
                    st.markdown(f"- {bullet}")

                st.markdown(f"**Risk Flags:** {', '.join(art.get('risk_flags', ['None']))}")

                with st.expander("🔍 Observability: Agent Trace, Sources & Fallbacks"):
                    for log in art.get("logs", []):
                        st.text(log)

        # --- Exportable Report ---
        if st.session_state.history:
            st.markdown("---")
            clean_history = []
            for entry in st.session_state.history:
                clean_entry = entry.copy()
                if "logs" in clean_entry:
                    clean_entry["logs"] = [msg.content if hasattr(msg, 'content') else str(msg) for msg in clean_entry["logs"]]
                clean_history.append(clean_entry)

            json_report = json.dumps(clean_history, indent=4)
            st.download_button(
                label="💾 Download Audit Report (JSON)",
                data=json_report,
                file_name=f"FinAgent_Audit_{datetime.datetime.now().strftime('%Y%m%d')}.json",
                mime="application/json",
                use_container_width=True
            )

# --- TAB 2: RAG DEBUGGER INTERFACE ---
with tab_debug:
    st.subheader("🔍 Knowledge Base Search & Evaluation")
    st.markdown("Query the FAISS vector database directly to inspect retrieved chunks and evaluate embedding quality.")

    search_query = st.text_input("Enter search query (e.g., 'What are the margin risks for NVDA?'):")
    num_results = st.slider("Number of chunks to retrieve (k):", min_value=1, max_value=10, value=3)

    if st.button("🔎 Run Vector Search", type="primary"):
        if st.session_state.vector_store is None:
            st.error("⚠️ The Vector Store is empty. Please upload and ingest a document first using the sidebar.")
        elif not search_query:
            st.warning("Please enter a search query.")
        else:
            with st.spinner("Searching FAISS database..."):
                docs = st.session_state.vector_store.similarity_search(search_query, k=num_results)

                if not docs:
                    st.warning("No relevant chunks found.")
                else:
                    st.success(f"Retrieved {len(docs)} document chunks.")
                    for i, doc in enumerate(docs):
                        with st.expander(f"📄 Result {i+1} | Source: {doc.metadata.get('source', 'Unknown')}", expanded=True):
                            st.write(doc.page_content)

# --- TAB 3: CONVERSATIONAL INTERFACE (BONUS FEATURE) ---
with tab_chat:
    st.subheader("💬 Chat with FinAgent")
    st.markdown("Ask follow-up questions to challenge the AI's thesis on a specific generated artefact.")

    if not st.session_state.history:
        st.info("No decisions generated yet. Run the pipeline on the Live Dashboard first!")
    else:
        # Create a dictionary to easily reference historical decisions by a friendly dropdown name
        options = {f"{art['asset']} - {art['action']} ({art['timestamp']})": art for art in st.session_state.history}
        selected_option = st.selectbox("Select a decision to discuss:", list(options.keys()))
        selected_art = options[selected_option]

        # Unique chat memory key for the specific dropdown selection so contexts don't mix
        chat_key = f"chat_{selected_option}"
        if chat_key not in st.session_state:
            st.session_state[chat_key] = []

        # Draw existing chat history
        for msg in st.session_state[chat_key]:
            with st.chat_message(msg["role"]):
                st.markdown(msg["content"])

        # Chat Input Box
        if prompt_text := st.chat_input(f"e.g., Why did you rate {selected_art['asset']} as {selected_art['confidence_level']} confidence?"):

            # Display user message
            st.session_state[chat_key].append({"role": "user", "content": prompt_text})
            with st.chat_message("user"):
                st.markdown(prompt_text)

            # Process AI Response
            with st.chat_message("assistant"):
                with st.spinner("Analyzing artefact history..."):
                    # We inject the exact context of the specific decision into the LLM prompt!
                    system_prompt = f"""You are the quantitative analyst who just recommended to {selected_art['action']} {selected_art['asset']}.
                    You evaluated it at a price of {selected_art['price_evaluated']} and gave it a confidence score of {selected_art['confidence_score']}/100.

                    Here was your internal hypothesis: "{selected_art.get('internal_hypothesis', 'None recorded.')}"
                    Here was your main evidence: {', '.join(selected_art['evidence'])}
                    Here were the risks you flagged: {', '.join(selected_art['risk_flags'])}

                    A portfolio manager is asking you a follow-up question. Answer concisely, professionally, and defend your thesis based on the facts provided above."""

                    chat_prompt_template = ChatPromptTemplate.from_messages([
                        ("system", system_prompt),
                        ("user", "{user_input}")
                    ])

                    chain = chat_prompt_template | llm
                    response = chain.invoke({"user_input": prompt_text})

                    st.markdown(response.content)

            # Save AI response to chat history
            st.session_state[chat_key].append({"role": "assistant", "content": response.content})

Overwriting app.py


In [ ]:
!pip install -qU langchain-groq langgraph langchain-core pydantic

In [44]:
import time

!pkill -9 -f streamlit
!pkill -9 -f ssh

print("Starting Streamlit server...")
!nohup streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1 &

time.sleep(5)

print("Starting Tunnel...")
!ssh -o StrictHostKeyChecking=no -T -p 443 -R0:localhost:8501 a.pinggy.io

Starting Streamlit server...
Starting Tunnel...
Allocated port 4 for remote forward to localhost:8501
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
http://shkeg-34-106-255-221.free.pinggy.net
http://thfkm-34-106-255-221.run.pinggy-free.link
https://shkeg-34-106-255-221.free.pinggy.net
https://thfkm-34-106-255-221.run.pinggy-free.link
RB: 95878, SB: 3957490, TC: 19, AC: 1               


Time exceeded



Connection to a.pinggy.io closed by remote host.
